# 🔄 第4章：查询循环与 API 交互

## 🎯 学习目标
1. 理解 query.ts 的核心循环逻辑
2. 理解流式响应的处理机制
3. 理解工具并发执行 (StreamingToolExecutor)
4. 理解自动压缩 (auto-compact) 机制
5. 理解 Token 预算和成本追踪

---

## 4.1 query() 函数概览

`query.ts` (67KB, 1730行) 是整个项目最核心的文件之一。
它实现了 **LLM 调用 → 工具执行 → 再调用** 的核心循环。

### 函数签名

```typescript
export async function* query(params: QueryParams)
  : AsyncGenerator<StreamEvent | Message | ToolUseSummaryMessage, Terminal>
```

**注意返回类型：**
- `AsyncGenerator` — 异步生成器，逐条产出消息
- `StreamEvent` — 流式事件 (message_start, content_block_delta 等)
- `Message` — 完整消息 (assistant, user, system)
- `Terminal` — 终止状态 (成功/失败/中断)

### 核心参数

```typescript
type QueryParams = {
  messages: Message[];           // 消息历史
  systemPrompt: SystemPrompt;    // 系统提示词
  userContext: { [k: string]: string };  // 用户上下文 (CLAUDE.md 等)
  systemContext: { [k: string]: string }; // 系统上下文 (git status 等)
  canUseTool: CanUseToolFn;      // 权限检查函数
  toolUseContext: ToolUseContext; // 工具执行上下文
  fallbackModel?: string;        // 备用模型
  querySource: QuerySource;      // 查询来源 (repl/sdk/agent)
  maxTurns?: number;             // 最大轮次
  taskBudget?: { total: number }; // Token 预算
};
```

## 4.2 查询循环的状态机

query() 内部是一个状态机，每次迭代可能产生两种结果：

```typescript
// 终止状态
type Terminal = { type: 'terminal'; reason: string };

// 继续状态
type Continue = { type: 'continue'; reason: string };
```

### 循环伪代码

```typescript
async function* queryLoop(params) {
  let state = initializeState(params);
  
  while (true) {
    // ===== Phase 1: 准备 API 请求 =====
    const config = buildQueryConfig();
    const messagesForAPI = normalizeMessagesForAPI(state.messages);
    
    // 注入上下文
    prependUserContext(messagesForAPI, userContext);
    appendSystemContext(messagesForAPI, systemContext);
    
    // ===== Phase 2: 调用 Claude API =====
    const stream = await callClaudeAPI({
      messages: messagesForAPI,
      system: systemPrompt,
      tools: toolDefinitions,
      model: mainLoopModel,
    });
    
    // ===== Phase 3: 处理流式响应 =====
    let assistantMessage;
    for await (const event of stream) {
      yield event;  // 转发流式事件给上层
      
      if (event.type === 'message_start') { /* 初始化 */ }
      if (event.type === 'content_block_start') {
        if (event.content_block.type === 'tool_use') {
          // 开始流式工具执行！
          streamingToolExecutor.startTool(event.content_block);
        }
      }
      if (event.type === 'content_block_stop') {
        assistantMessage = buildAssistantMessage(event);
        yield assistantMessage;
      }
    }
    
    // ===== Phase 4: 执行工具 =====
    if (hasToolUseBlocks(assistantMessage)) {
      const toolResults = yield* runTools(
        assistantMessage,
        toolUseContext,
        canUseTool,
      );
      
      // 将 tool_result 追加到消息
      state.messages.push(...toolResults);
      state.turnCount++;
      
      // 检查是否超过最大轮次
      if (maxTurns && state.turnCount >= maxTurns) {
        yield maxTurnsReachedAttachment;
        return { type: 'terminal', reason: 'max_turns' };
      }
      
      continue; // 继续循环
    }
    
    // ===== Phase 5: 停止条件 =====
    if (assistantMessage.stop_reason === 'end_turn') {
      // 执行 Stop Hooks
      yield* handleStopHooks(...);
      return { type: 'terminal', reason: 'end_turn' };
    }
  }
}
```

## 4.3 流式工具执行 (StreamingToolExecutor)

Claude Code 的一个创新是 **流式工具执行**：
当 Claude 返回多个 tool_use 块时，不等所有块接收完毕，
而是在接收到每个块的完整输入后立即开始执行。

```
时间线：

Claude API 流式返回:
  ├── tool_use[0]: Glob { pattern: '**/*.ts' }     ← 接收完毕
  │   └── 立即开始执行 Glob！                        ← 并行执行
  ├── tool_use[1]: Grep { pattern: 'TODO' }         ← 接收完毕
  │   └── 立即开始执行 Grep！                        ← 并行执行
  └── tool_use[2]: Read { file: 'README.md' }       ← 接收完毕
      └── 立即开始执行 Read！                        ← 并行执行

所有工具并行执行，最终结果按顺序返回。
```

### 并发安全检查

```typescript
// 只有 isConcurrencySafe() 返回 true 的工具才能并行
if (tool.isConcurrencySafe(input)) {
  // 并行执行
  executeInParallel(tool, input);
} else {
  // 串行执行，等待前面的工具完成
  await previousToolsComplete();
  executeSequentially(tool, input);
}
```

**哪些工具可以并行？**
- ✅ Glob, Grep, Read, WebSearch, WebFetch — 只读操作
- ❌ Bash, Edit, Write, Agent — 可能修改状态

## 4.4 自动压缩 (Auto-Compact)

当对话变得很长时，消息历史会超过模型的上下文窗口。
Auto-compact 机制自动压缩历史消息：

```
触发条件：
  当前 token 数 > 上下文窗口 × 阈值 (通常 80%)

压缩流程：
  1. 选择要压缩的消息范围
  2. 调用 Claude API 生成摘要
  3. 用摘要替换原始消息
  4. 插入 compact_boundary 标记
  5. 继续正常对话

消息历史变化：
  Before: [msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]
  After:  [compact_boundary, summary_msg, msg7, msg8]
```

### Token 警告状态

```typescript
type AutoCompactTrackingState = {
  warningState: 'none' | 'approaching' | 'critical';
  lastContextTokens: number;
  compactCount: number;
};

// 状态转换
// none → approaching: 接近上下文窗口限制
// approaching → critical: 即将超出
// critical → 触发 auto-compact
```

## 4.5 消息规范化 (normalizeMessagesForAPI)

在发送给 API 之前，消息需要经过规范化处理：

```typescript
function normalizeMessagesForAPI(messages: Message[]): MessageParam[] {
  // 1. 过滤掉内部消息类型
  //    - progress 消息
  //    - attachment 消息
  //    - system local_command 消息
  
  // 2. 处理 compact_boundary
  //    只保留 boundary 之后的消息
  
  // 3. 处理 thinking 块
  //    thinking 块必须在 assistant 消息中
  //    且不能是最后一个块
  
  // 4. 合并连续的同类型消息
  //    API 要求 user/assistant 交替出现
  
  // 5. 应用 tool result budget
  //    超大的 tool_result 会被截断或持久化到磁盘
  
  return normalizedMessages;
}
```

### 上下文注入

```typescript
// 用户上下文注入到第一条用户消息之前
prependUserContext(messages, {
  claudeMd: '# Project Rules\n...',
  currentDate: "Today's date is 2026-03-31.",
});

// 系统上下文追加到最后一条用户消息之后
appendSystemContext(messages, {
  gitStatus: 'Current branch: main\nStatus: (clean)\n...',
});
```

## 4.6 错误恢复机制

query() 内置了多种错误恢复机制：

### max_output_tokens 恢复

```typescript
const MAX_OUTPUT_TOKENS_RECOVERY_LIMIT = 3;

// 当 Claude 的响应被截断（达到 max_output_tokens）时：
// 1. 记录当前状态
// 2. 增加 max_output_tokens
// 3. 重新发送请求
// 4. 最多重试 3 次
```

### prompt_too_long 恢复

```typescript
// 当消息历史太长导致 prompt_too_long 错误时：
// 1. 触发 reactive compact（响应式压缩）
// 2. 压缩历史消息
// 3. 重新发送请求
// 4. 只尝试一次
```

### API 重试

```typescript
// 网络错误、速率限制等可重试错误：
// 使用 withRetry() 包装 API 调用
// 指数退避 + 抖动
// 支持 fallback model（备用模型）
```

## 4.7 成本追踪 (cost-tracker.ts)

```typescript
// 追踪每次 API 调用的 token 使用量
type NonNullableUsage = {
  input_tokens: number;       // 输入 token
  output_tokens: number;      // 输出 token
  cache_creation_input_tokens: number;  // 缓存创建
  cache_read_input_tokens: number;      // 缓存读取
};

// 在 QueryEngine 中累积
this.totalUsage = accumulateUsage(this.totalUsage, currentMessageUsage);

// 检查预算
if (maxBudgetUsd !== undefined && getTotalCost() >= maxBudgetUsd) {
  yield { type: 'result', subtype: 'error_max_budget_usd', ... };
  return;
}
```

### Prompt Cache 优化

Claude Code 大量使用 Anthropic 的 prompt cache：

```
System Prompt (缓存)  ← 跨请求共享
  + Tool Definitions (缓存) ← 排序稳定，最大化命中
  + User Context (缓存)     ← 会话内不变
  + Message History          ← 增量增长
  + New User Message         ← 每次不同

缓存命中率越高，成本越低，延迟越小。
这就是为什么 tools.ts 中要对工具列表排序！
```

## 4.8 Stop Hooks

当 Claude 决定停止（stop_reason = 'end_turn'）时，
系统会执行 Stop Hooks 来决定是否真的停止：

```typescript
// query/stopHooks.ts
async function* handleStopHooks(messages, context) {
  // 1. 执行用户定义的 Stop hooks
  for (const hook of stopHooks) {
    const result = await executeHook(hook, { messages });
    
    if (result.action === 'continue') {
      // Hook 说继续！注入额外消息
      yield createUserMessage({ content: result.message });
      return 'continue';  // 不停止，继续循环
    }
  }
  
  // 2. 所有 hooks 都同意停止
  return 'stop';
}
```

**用例：** 自动化测试场景中，Stop Hook 可以检查测试是否通过，
如果没通过就让 Claude 继续修复。

## 4.9 消息队列与命令注入

在查询循环运行期间，用户可以通过消息队列注入新的命令：

```typescript
// 在每次循环迭代开始时检查队列
const queuedCommands = getCommandsByMaxPriority();

if (queuedCommands.length > 0) {
  for (const cmd of queuedCommands) {
    // 处理排队的命令
    if (isSlashCommand(cmd)) {
      // 执行 /slash 命令
    } else {
      // 注入为新的用户消息
      messages.push(createUserMessage({ content: cmd.prompt }));
    }
    removeFromQueue(cmd.uuid);
  }
}
```

这允许用户在 Claude 工作时发送新的指令（如 "btw, 也修复一下那个 bug"）。

## ✅ 本章小结

| 概念 | 关键点 |
|------|--------|
| query() 循环 | API 调用 → 流式接收 → 工具执行 → 继续/停止 |
| 流式工具执行 | 接收到完整输入后立即执行，并发安全的工具可并行 |
| Auto-compact | 上下文超限时自动压缩历史消息 |
| 消息规范化 | 过滤内部消息、处理 thinking、合并连续消息 |
| 错误恢复 | max_output_tokens 重试、prompt_too_long 压缩、API 重试 |
| 成本追踪 | Token 使用量累积、预算检查、prompt cache 优化 |
| Stop Hooks | 用户自定义的停止条件检查 |

### 下一章预告
第5章将深入 **多 Agent 系统**——理解 AgentTool 如何创建和管理子 Agent。